In [1]:
!cd /content && python bench2.py --interleave --iters 40 --dtype float16 --systems "ordinary,suffixkv,incgraph[region],incgraph[region+packed]"

device=cuda dtype=float16 B=1 T=2048 C=128 (N=16 chunks) D=768 H=12 L=12
gpu: Tesla T4 | torch 2.11.0+cu130
verifying correctness (small model) ... ok
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(

== edit-first
system                         wall p50 [p10-p90] ms  speedup  host ms  gpuspan kern/step  kern ms memcpy syncs  nodes checks bcalls  peak MiB cache MiB
ordinary                         25.48 [25.04-26.01]      1.00x     3.33        -       120    23.68   13.0   0.0      -      -      -         -         0
suffixkv                         26.73 [26.20-27.12]      0.95x     4.08        -       145    24.31   13.0   0.0      -      -      -         -        72
incgraph[region]                 26.97 [26.64-27.67]      0.94x     5.08        -       156    24.72   25.0   0.

In [2]:
!pip -q install datasets transformers
!python edit_trace_bench.py

[dataset] Loading real Python commit traces from: https://huggingface.co/datasets/bigcode/commitpackft/resolve/main/data/python/data.jsonl
config.json: 100% 665/665 [00:00<00:00, 3.34MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 87.8kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 1.95MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 993kB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 1.73MB/s]
[dataset] SUCCESS: real commit traces loaded; edits=200, context_tokens=8192
dataset=commitpackft-python-jsonl  edits=200  N=16 chunks of 128 tokens
median edited chunk p=5.0/16   mean recompute fraction=0.619   (=> at most 1.61x fewer chunk-executions than full recompute)
correctness: all systems match ordinary (checked every 25 steps + final)

system                          mean ms     p50  speedup   by edit position (first/mid/last third)
ordinary                          25.23   25.33    1.00x   1.00x / 1.00x / 1.00x
suffixkv                          18.71   20.24    1.35x   1.